In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# RGB-DCT fixed T49 VAE-lift development trial

Run all once. Two fixed new prompt/seed pairs produce eight planned MP4 slots. The direct RGB +/- controls gate one positive VAE-lift T49 native arm per source. Continuous actual-MP4 scores are persisted before same-source comparisons. Zero lift/response are method negatives. This is development controllability only: no binary detection, low FPR, quality, payload, attribution, or multi-step gain claim. The user performs the GPU/Colab run.


In [ ]:
from pathlib import Path
import datetime, json, sys
SOURCE_SHA = 'a883659010870e613deaa612419cbe2630ec36a3'
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/RGB-DCT-T49-VAE-Lift-V1')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / stamp
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(status='SETUP_STARTED', source_sha=SOURCE_SHA, output_dir=str(OUTPUT), python=sys.version, executable=sys.executable), indent=2) + '\n', encoding='utf-8')
print('fresh output:', OUTPUT, flush=True)


In [ ]:
import subprocess, sys, json
SETUP_LOG = OUTPUT / 'setup.log'
def logged_run(command, *, cwd=None, env=None, check=True):
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True); log.write(line); log.flush()
        child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in child.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n'); log.flush()
    if check and returncode:
        (OUTPUT / 'setup_failure.json').write_text(json.dumps(dict(command=command, returncode=returncode), indent=2) + '\n', encoding='utf-8')
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)
import importlib.metadata, subprocess, sys
print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
logged_run([sys.executable, '-m', 'pip', '--version'], check=True)
logged_run(['apt-get', 'update', '-qq'], check=True)
logged_run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
print('torch before install:', version('torch'), flush=True)
if (version('torch') or '').split('+', 1)[0] != '2.11.0':
    logged_run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
logged_run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
check_code = """
import importlib.metadata, sys, torch, diffusers
from diffusers import WanPipeline, AutoencoderKLWan
print('Fresh process Python:', sys.version, flush=True)
print('Fresh process executable:', sys.executable, flush=True)
for name in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):
    try:
        value = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        value = None
    print(name + ':', value, flush=True)
assert str(torch.__version__).split('+', 1)[0] == '2.11.0', torch.__version__
assert torch.cuda.is_available(), 'CUDA torch required'
assert diffusers.__version__ == '0.40.0', diffusers.__version__
"""
logged_run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
REPO = Path('/content/SC-SSTW-RGB-DCT-T49-' + stamp)
logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
logged_run(['git', '-C', str(REPO), 'fetch', 'origin', 'dev/rgb-dct-temporal-balanced-v1'])
logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError('immutable source SHA readback mismatch')
(OUTPUT / 'source_receipt.json').write_text(json.dumps(dict(expected_sha=SOURCE_SHA, actual_sha=actual, repo=str(REPO)), indent=2) + '\n', encoding='utf-8')
print('source commit:', actual, flush=True)


In [ ]:
import importlib.metadata, shutil, torch, numpy, diffusers
environment_receipt = dict(
    ffmpeg=shutil.which('ffmpeg'), ffprobe=shutil.which('ffprobe'),
    python=sys.version, torch=torch.__version__, torch_cuda_runtime=torch.version.cuda,
    numpy=numpy.__version__, diffusers=diffusers.__version__,
    cuda_available=torch.cuda.is_available(),
    device=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
)
(OUTPUT / 'environment_receipt.json').write_text(json.dumps(environment_receipt, indent=2) + '\n', encoding='utf-8')
if not environment_receipt['ffmpeg'] or not environment_receipt['ffprobe'] or not environment_receipt['cuda_available']:
    raise RuntimeError('ffmpeg, ffprobe, and a CUDA runtime are required for this fixed Wan run')
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(status='SETUP_COMPLETE', source_sha=SOURCE_SHA, output_dir=str(OUTPUT), python=sys.version, executable=sys.executable), indent=2) + '\n', encoding='utf-8')
print('fixed Wan environment:', environment_receipt, flush=True)


In [ ]:
import os
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.rgb_dct_t49_vae_lift_run', '--output', str(OUTPUT)]
completed = logged_run(command, cwd=REPO, env=env, check=False)
RESULT_PATH = OUTPUT / 'result.json'
(OUTPUT / 'execution_receipt.json').write_text(json.dumps(dict(command=command, returncode=completed.returncode, result_path=str(RESULT_PATH)), indent=2) + '\n', encoding='utf-8')
if not RESULT_PATH.exists():
    raise FileNotFoundError('runner produced no retained result.json')


In [ ]:
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
if result['fixed_denominator'] != {'sources': 2, 'mp4_score_slots': 8, 'frames': 1448}:
    raise RuntimeError('fixed denominator mismatch')
print('status:', result['status'], flush=True)
print('attempted/scored/invalid/pending:', result['attempted_media_slots'], result['scored_media_slots'], result['invalid_media_slots'], result['pending_media_slots'], flush=True)
for case_id, case in result['cases'].items():
    for arm, row in case['slots'].items():
        print(case_id, arm, row['status'], row['score'], row['reason'], flush=True)
print('full result:', RESULT_PATH, flush=True)
